# Assignment 3 — Shading, Textures, Bumps

> **GAMES101 — Intro to Computer Graphics** (Lingqi Yan, UCSB).
> Course site: <https://sites.cs.ucsb.edu/~lingqi/teaching/games101.html>
>
> The course ships C++ starter code with Eigen + OpenCV. I'm doing the same tasks in
> Python notebooks so I can iterate on the math cell-by-cell. Notes at the top of each
> notebook are what I actually needed to remember to get the assignment out.

## Topic

This is the payoff assignment for the rasterisation block. Five shaders on the
same cow mesh:

1. **Normal shader** — visualise surface normals as RGB (sanity check).
2. **Blinn-Phong** — ambient + diffuse + specular. Specular uses the *half-vector*
   `h = normalize(l + v)` and raises `(n·h)` to a shininess exponent `p`.
3. **Texture shader** — sample albedo from a texture instead of a constant.
4. **Bump shader** — perturb the normal using a height map's local gradient.
5. **Displacement shader** — actually move the shading point along the normal.

![phong components](https://upload.wikimedia.org/wikipedia/commons/6/6b/Phong_components_version_4.png)
*Ambient + diffuse + specular = final. (Wikipedia)*

See: [Blinn–Phong reflection model](https://en.wikipedia.org/wiki/Blinn%E2%80%93Phong_reflection_model), [Phong reflection model](https://en.wikipedia.org/wiki/Phong_reflection_model), [Bump mapping](https://en.wikipedia.org/wiki/Bump_mapping).


In [1]:
import numpy as np

def normal_shader(payload):
    n = payload['normal']
    return (n * 0.5 + 0.5) * 255.0


In [2]:
def blinn_phong(payload, lights, eye_pos):
    ka = np.array([0.005, 0.005, 0.005])
    kd = payload['color']
    ks = np.array([0.7937, 0.7937, 0.7937])
    p = 150.0
    amb_intensity = np.array([10, 10, 10])
    n = payload['normal'] / np.linalg.norm(payload['normal'])
    point = payload['pos']
    v = (eye_pos - point) / np.linalg.norm(eye_pos - point)
    result = ka * amb_intensity
    for L in lights:
        l_vec = L['pos'] - point
        r2 = float(l_vec @ l_vec)
        l = l_vec / np.sqrt(r2)
        h = (l + v) / np.linalg.norm(l + v)
        diff = kd * (L['I'] / r2) * max(0.0, float(n @ l))
        # specular -- forgot the exponent below, dim highlights
        spec = ks * (L['I'] / r2) * max(0.0, float(n @ h))
        result += diff + spec
    return np.clip(result * 255, 0, 255)


Specular highlights look weak/spread out. Must be the missing `** p` on the half-vector term.